# LeetCode 670 — Maximum Swap bằng Python

Link đề: https://leetcode.com/problems/maximum-swap/ · Độ khó: **Medium** · Chủ đề: Greedy (tham lam), Math, thao tác chữ số

Skeleton LeetCode:

```python
class Solution:
    def maximumSwap(self, num: int) -> int:
        ...
```

Ghi chú: đề cho phép **đổi chỗ hai chữ số tối đa một lần** để được số lớn nhất. Cái hay của bài là câu hỏi "đổi chỗ **hai chữ số nào**?" có đáp án tham lam gọn: *nâng chữ số lớn nhất có thể lên vị trí cao nhất có thể*. Notebook đi từ thử mọi cặp vị trí, tới cách duyệt bằng mảng "vị trí xuất hiện cuối", mảng "chỉ số lớn nhất bên phải", một lượt duyệt từ phải sang trái, và cách so với dãy chữ số đã sắp xếp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được đổi chỗ **hai chữ số bất kỳ** của `num` **tối đa một lần** (có thể không đổi), tìm số lớn nhất thu được.
2. Nắm các công cụ Python cần dùng: chuyển số ↔ chuỗi (`str`, `int`), `list(s)` để sửa được từng ký tự, đổi chỗ bằng `a[i], a[j] = a[j], a[i]`, `enumerate`, `range` đi ngược, `str.rindex`, `itertools.combinations`, `sorted(..., reverse=True)`.
3. Đi từ **thử mọi cặp vị trí** → **bảng vị trí cuối của từng chữ số** → **mảng chỉ số lớn nhất bên phải** → **một lượt duyệt phải-sang-trái** → **so với dãy chữ số sắp xếp giảm**, và hiểu vì sao các cách cho cùng đáp án.
4. Hiểu mấu chốt: chữ số ở vị trí **càng bên trái càng có giá trị lớn**; vì vậy ta đổi chỗ ở vị trí **trái nhất** mà còn "nâng" được, và chọn chữ số **lớn nhất** (nếu trùng thì lấy cái **ngoài cùng bên phải**) từ bên phải đem lên.
5. Tránh các lỗi kinh điển: chọn lần xuất hiện **đầu tiên** thay vì **cuối cùng** của chữ số lớn nhất, sắp xếp giảm dần toàn bộ (nhiều hơn một lần đổi), đổi tiếp sau lần đổi đầu, chỉ so với chữ số lớn nhất toàn cục.

## 2. Đề bài & ví dụ

Cho một số nguyên `num`. Bạn có thể **đổi chỗ hai chữ số** của nó **tối đa một lần** để được số có giá trị lớn nhất. Trả về số lớn nhất đó.

**Ví dụ 1**

- Input: `num = 2736`
- Output: `7236`
- Giải thích: đổi chỗ chữ số `2` và chữ số `7`.

**Ví dụ 2**

- Input: `num = 9973`
- Output: `9973`
- Giải thích: không cần đổi chỗ (đã là lớn nhất có thể).

**Ràng buộc:**

- `0 <= num <= 10^8`

Hệ quả: `num` có tối đa 9 chữ số (`10^8 = 100000000`), nên rất nhỏ. Cách thử mọi cặp cũng chạy nhanh được, nhưng ta vẫn học cách làm gọn, vì đây là mẫu tư duy tham lam quan trọng.

In [1]:
from typing import List
from itertools import combinations
import random

examples = [
    (2736, 7236),
    (9973, 9973),
]
for num, expected in examples:
    print(f"num = {num} | số chữ số = {len(str(num))} | kỳ vọng {expected}")

num = 2736 | số chữ số = 4 | kỳ vọng 7236
num = 9973 | số chữ số = 4 | kỳ vọng 9973


## 3. Kiến thức nền

### 3.1. Chuyển số ↔ chuỗi để làm việc với từng chữ số

`str(2736)` cho chuỗi `"2736"`; từng ký tự là một chữ số (dạng chữ). `int("7236")` đổi ngược lại thành số. Có thể lấy chữ số bằng chỉ số: `"2736"[0]` là `"2"`.

Lưu ý: ký tự là chữ, không phải số: `"2" + "7"` là `"27"`, không phải `9`. Muốn dùng như số thì `int("2")`. Tuy vậy **so sánh** hai ký tự chữ số (`"7" > "2"`) cho kết quả giống so sánh số, vì mã ký tự của `"0".."9"` tăng dần. Điều này cho phép ta so sánh chữ số ngay trên chuỗi.

In [2]:
num = 2736
s = str(num)
print(s, type(s).__name__)       # '2736' str
print(s[0], s[-1])               # '2' '6'
print("2" + "7")                 # '27' (nối chuỗi), KHÔNG phải 9
print(int("2") + int("7"))       # 9
print("7" > "2", "3" < "6")     # True True: so sánh TỪNG chữ số bằng ký tự vẫn đúng thứ tự số
print(int("7236"))               # 7236

2736 str
2 6
27
9
True True
7236


### 3.2. `list(s)` và đổi chỗ hai phần tử

Chuỗi trong Python **không sửa được** tại chỗ (`s[0] = "7"` báo lỗi). Ta chuyển sang `list` để sửa, rồi `"".join(...)` để ghép lại.

Đổi chỗ hai phần tử không cần biến tạm: `a[i], a[j] = a[j], a[i]` (Python tính vế phải trước rồi mới gán).

In [3]:
s = "2736"
try:
    s[0] = "7"
except TypeError as e:
    print("chuỗi không sửa được:", e)

a = list(s)
print(a)                          # ['2', '7', '3', '6']
a[0], a[1] = a[1], a[0]
print(a, "->", "".join(a))        # ['7', '2', '3', '6'] -> 7236
print(int("".join(a)))

chuỗi không sửa được: 'str' object does not support item assignment
['2', '7', '3', '6']
['7', '2', '3', '6'] -> 7236
7236


### 3.3. `enumerate`, `range` đi ngược, và `str.rindex`

- `enumerate(s)` cho từng cặp `(chỉ số, ký tự)`.
- `range(n - 1, -1, -1)` đi từ `n - 1` về `0` (duyệt phải sang trái).
- `s.rindex(ch)` trả về chỉ số xuất hiện **cuối cùng** của `ch` (còn `s.index(ch)` là xuất hiện **đầu tiên**). Hai hàm này hay bị nhầm; bài này cần **lần cuối**.

In [4]:
s = "1993"
for i, ch in enumerate(s):
    print(i, ch, end=" | ")
print()
print(list(range(len(s) - 1, -1, -1)))     # [3, 2, 1, 0]
print("index đầu của '9':", s.index("9"), "| rindex (cuối) của '9':", s.rindex("9"))

0 1 | 1 9 | 2 9 | 3 3 | 
[3, 2, 1, 0]
index đầu của '9': 1 | rindex (cuối) của '9': 2


### 3.4. `itertools.combinations` để thử mọi cặp vị trí

`combinations(range(n), 2)` sinh mọi cặp `(i, j)` với `i < j` — đúng "mọi cặp vị trí để đổi chỗ" (đổi `i` với `j` và `j` với `i` là một). Với `n = 4` có `C(4, 2) = 6` cặp.

In [5]:
print(list(combinations(range(4), 2)))
print(len(list(combinations(range(9), 2))), "cặp cho số 9 chữ số")

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
36 cặp cho số 9 chữ số


### 3.5. `sorted(..., reverse=True)`

`sorted("2736", reverse=True)` trả về danh sách ký tự xếp giảm dần: `['7', '6', '3', '2']`. Đó là số **lớn nhất có thể ghép từ các chữ số đó** (nhưng thường cần đổi nhiều hơn một lần, nên không phải đáp án trực tiếp — chỉ dùng để so sánh).

In [6]:
d = sorted("2736", reverse=True)
print(d, "->", "".join(d))          # 7632 (cần nhiều hơn một lần đổi chỗ)
print(sorted("9973", reverse=True))  # ['9', '9', '7', '3'] đúng bằng '9973' -> không cần đổi

['7', '6', '3', '2'] -> 7632
['9', '9', '7', '3']


## 4. Ý tưởng nền tảng / trực giác

So sánh hai số cùng số chữ số: **chữ số khác nhau đầu tiên từ trái sang** quyết định số nào lớn hơn. Vì vậy ta muốn chữ số **bên trái nhất** càng lớn càng tốt.

Tính tay `2736`:

- Vị trí 0 đang là `2`. Bên phải có `7, 3, 6`; có chữ số lớn hơn `2` (là `7`). Đổi `2` với `7` → `7236`: chữ số đầu từ `2` lên `7`, tăng rất nhiều.
- Nếu đổi `2` với `6` → `6732`; đổi `2` với `3` → `3726`. Tất cả nhỏ hơn `7236`.

Tính tay `9973`: vị trí 0 là `9`, không có chữ số nào lớn hơn `9` ở bên phải. Vị trí 1 là `9`, tương tự. Vị trí 2 là `7`: bên phải chỉ có `3` nhỏ hơn. Vị trí 3: không còn gì bên phải. Không đổi được gì tốt hơn → giữ nguyên `9973`.

Quy luật: duyệt từ trái sang, ở **vị trí đầu tiên** mà bên phải có chữ số **lớn hơn** thì đổi, và đổi với chữ số lớn nhất bên phải.

In [7]:
s = "2736"
for i in range(len(s)):
    for j in range(i + 1, len(s)):
        t = list(s)
        t[i], t[j] = t[j], t[i]
        print(f"đổi vị trí {i} và {j}: {s} -> {''.join(t)}")

đổi vị trí 0 và 1: 2736 -> 7236
đổi vị trí 0 và 2: 2736 -> 3726
đổi vị trí 0 và 3: 2736 -> 6732
đổi vị trí 1 và 2: 2736 -> 2376
đổi vị trí 1 và 3: 2736 -> 2637
đổi vị trí 2 và 3: 2736 -> 2763


## 5. Cách 1 — Brute Force: thử mọi cặp vị trí

### Ý tưởng

Chỉ có `C(d, 2)` cách đổi chỗ (với `d` là số chữ số) cộng thêm "không đổi". Thử tất cả, giữ số lớn nhất.

1. Đổi `num` thành chuỗi `s`.
2. `best = num` (trường hợp không đổi).
3. Với mọi cặp `i < j`: đổi chỗ trên bản sao của `s`, chuyển thành số, cập nhật `best`.

### Code

In [8]:
def maximum_swap_brute(num):
    s = str(num)
    best = num                                   # không đổi chỗ
    for i, j in combinations(range(len(s)), 2):
        t = list(s)
        t[i], t[j] = t[j], t[i]
        best = max(best, int("".join(t)))
    return best

for num, expected in examples:
    got = maximum_swap_brute(num)
    print(got, expected, "OK" if got == expected else "SAI")

7236 7236 OK
9973 9973 OK


### Độ phức tạp và nút thắt

- **Time:** `O(d^3)` — `O(d^2)` cặp, mỗi cặp copy và ghép chuỗi `O(d)` (với `d ≤ 9` rất nhỏ). Nếu `num` có `d` chữ số lớn thì chậm.
- **Space:** `O(d)` cho bản sao.

**Nút thắt:** với số hàng nghìn chữ số, `d^3` không chấp nhận được; và ta đang thử cả những cặp chắc chắn không tối ưu (ví dụ đổi hai chữ số cuối khi đầu số còn nâng được). Cần một quy tắc chọn thẳng cặp tối ưu. Brute force vẫn rất quý làm **oracle** để kiểm tra các cách tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

Chứng minh bằng lời cho quy tắc tham lam:

1. **Vị trí cần đổi:** gọi `i` là vị trí trái nhất mà bên phải `i` có chữ số **lớn hơn** `s[i]`. Nếu không có vị trí nào như vậy thì dãy chữ số **không tăng** từ trái sang phải (mỗi chữ số ≥ mọi chữ số bên phải) — đã tối ưu, không đổi.
2. **Chữ số đem lên:** tại `i`, chọn chữ số **lớn nhất** `M` ở bên phải `i`. Đặt `M` vào vị trí `i` làm tăng chữ số ở vị trí cao nhất có thể thay đổi, và tăng nhiều nhất.
3. **Trùng chữ số (rất dễ sai):** nếu `M` xuất hiện nhiều lần bên phải `i`, lấy lần xuất hiện **cuối cùng (ngoài cùng bên phải)**. Lý do: chữ số `s[i]` (nhỏ hơn `M`) sẽ bị đẩy xuống vị trí đó, và ta muốn nó nằm ở vị trí **thấp nhất** để ít làm giảm số nhất.

Cell dưới minh hoạ bằng `1993`: `i = 0` (vì `1 < 9`), `M = 9` xuất hiện ở chỉ số 1 và 2. Đổi với chỉ số 2 → `9913`, đổi với chỉ số 1 → `9193`. `9913` lớn hơn.

In [9]:
s = "1993"
for j in (1, 2):
    t = list(s)
    t[0], t[j] = t[j], t[0]
    print(f"đổi vị trí 0 với {j}: {''.join(t)}")
print("đáp án brute:", maximum_swap_brute(1993))

đổi vị trí 0 với 1: 9193
đổi vị trí 0 với 2: 9913
đáp án brute: 9913


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bảng "vị trí cuối của mỗi chữ số 0..9"

**Giải thích từng đoạn.**

1. Tạo `last = [-1] * 10`; duyệt `s` từ trái sang, `last[int(ch)] = i` (ghi đè nên cuối cùng giữ **chỉ số lớn nhất** của chữ số `ch`).
2. Duyệt `i` từ trái sang. Với mỗi `i`, thử các chữ số `d` từ `9` giảm xuống `s[i] + 1`: nếu `last[d] > i` (có chữ số `d` ở **bên phải** `i`) thì đổi chỗ `i` với `last[d]` rồi dừng.
3. Vì thử `d` từ lớn xuống nhỏ nên lần đầu thoả mãn chính là chữ số lớn nhất bên phải; vì `last[d]` là lần xuất hiện cuối nên đúng yêu cầu "ngoài cùng bên phải".

**Code demo:**

In [10]:
def maximum_swap_last(num):
    s = list(str(num))
    last = [-1] * 10
    for i, ch in enumerate(s):
        last[int(ch)] = i                       # lần cuối cùng chữ số ch xuất hiện
    for i, ch in enumerate(s):
        for d in range(9, int(ch), -1):         # chữ số lớn hơn s[i], từ 9 trở xuống
            if last[d] > i:                     # có chữ số d ở bên phải i
                s[i], s[last[d]] = s[last[d]], s[i]
                return int("".join(s))
    return num                                  # không đổi được

for num, expected in examples:
    print(maximum_swap_last(num), expected)
print(maximum_swap_last(1993), maximum_swap_last(98368))   # 9913 98863

7236 7236
9973 9973
9913 98863


**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionLast:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        last = [-1] * 10
        for i, ch in enumerate(s):
            last[int(ch)] = i
        for i, ch in enumerate(s):
            for d in range(9, int(ch), -1):
                if last[d] > i:
                    s[i], s[last[d]] = s[last[d]], s[i]
                    return int("".join(s))
        return num

print(SolutionLast().maximumSwap(2736))   # 7236

7236


**Độ phức tạp (Cách 2):** Time `O(10 · d)` = `O(d)`, Space `O(d)` (danh sách chữ số) + `O(10)`.

### 7.2. Cách 3 — Mảng `d[i]` = chỉ số chữ số lớn nhất ở bên phải (kể cả `i`)

**Giải thích từng đoạn.**

1. Duyệt từ **phải sang trái**: `d[n-1] = n-1`. Với `i` từ `n-2` về `0`: nếu `s[i] > s[d[i+1]]` thì `d[i] = i`, ngược lại `d[i] = d[i+1]`.
2. Dấu `>` (không phải `>=`) đảm bảo khi **bằng nhau** ta giữ chỉ số **bên phải hơn** (`d[i+1]`) — đúng yêu cầu chọn lần xuất hiện cuối.
3. Duyệt `i` từ trái sang: gặp `s[i] < s[d[i]]` thì đổi `i` với `d[i]` và dừng.

**Code demo:**

In [12]:
def maximum_swap_rightmax(num):
    s = list(str(num))
    n = len(s)
    d = list(range(n))                          # d[i]: chỉ số chữ số lớn nhất trong s[i:], ưu tiên bên phải
    for i in range(n - 2, -1, -1):
        d[i] = i if s[i] > s[d[i + 1]] else d[i + 1]
    for i in range(n):
        if s[i] < s[d[i]]:
            s[i], s[d[i]] = s[d[i]], s[i]
            break
    return int("".join(s))

for num, expected in examples:
    print(maximum_swap_rightmax(num), expected)
print(maximum_swap_rightmax(1993), maximum_swap_rightmax(98368))

7236 7236
9973 9973
9913 98863


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionRightMax:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        n = len(s)
        d = list(range(n))
        for i in range(n - 2, -1, -1):
            d[i] = i if s[i] > s[d[i + 1]] else d[i + 1]
        for i in range(n):
            j = d[i]
            if s[i] < s[j]:
                s[i], s[j] = s[j], s[i]
                break
        return int("".join(s))

print(SolutionRightMax().maximumSwap(9973))   # 9973

9973


**Độ phức tạp (Cách 3):** Time `O(d)`, Space `O(d)` cho mảng `d` và danh sách chữ số.

### 7.3. Cách 4 — Một lượt duyệt phải-sang-trái, không mảng phụ

**Giải thích từng đoạn.** Ta không cần lưu cả mảng `d`: khi duyệt từ phải sang, chỉ cần nhớ `maxi` = chỉ số chữ số lớn nhất đã thấy bên phải (ưu tiên ngoài cùng bên phải), và cặp ứng viên `(l, r)`.

1. `maxi = n - 1`, `l = r = -1`.
2. Với `i` từ `n - 2` xuống `0`:
   - nếu `s[i] > s[maxi]`: chữ số này lớn hơn mọi thứ bên phải → `maxi = i` (không có gì để nâng ở `i`);
   - nếu `s[i] < s[maxi]`: đổi `i` với `maxi` sẽ nâng số → ghi nhận `l, r = i, maxi`. Vì ta duyệt về phía trái, lần ghi **sau cùng** là `i` **trái nhất**;
   - nếu bằng nhau: không làm gì (giữ `maxi` bên phải hơn).
3. Hết vòng lặp, nếu `l != -1` thì đổi `l` với `r`.

**Code demo:**

In [14]:
def maximum_swap_one_pass(num):
    s = list(str(num))
    n = len(s)
    maxi = n - 1
    l = r = -1
    for i in range(n - 2, -1, -1):
        if s[i] > s[maxi]:
            maxi = i
        elif s[i] < s[maxi]:
            l, r = i, maxi                      # ứng viên; i càng nhỏ (trái) càng ưu tiên
    if l != -1:
        s[l], s[r] = s[r], s[l]
    return int("".join(s))

for num, expected in examples:
    print(maximum_swap_one_pass(num), expected)
print(maximum_swap_one_pass(1993), maximum_swap_one_pass(98368))

7236 7236
9973 9973
9913 98863


**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionOnePass:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        n = len(s)
        maxi = n - 1
        l = r = -1
        for i in range(n - 2, -1, -1):
            if s[i] > s[maxi]:
                maxi = i
            elif s[i] < s[maxi]:
                l, r = i, maxi
        if l != -1:
            s[l], s[r] = s[r], s[l]
        return int("".join(s))

print(SolutionOnePass().maximumSwap(2736))   # 7236

7236


**Độ phức tạp (Cách 4):** Time `O(d)`, Space `O(d)` chỉ vì ta chuyển chuỗi sang danh sách; phần thuật toán dùng `O(1)` biến phụ.

### 7.4. Cách 5 — So với dãy chữ số sắp xếp giảm dần

**Giải thích từng đoạn.**

1. Gọi `t` là dãy chữ số của `num` **xếp giảm dần** (số lớn nhất có thể từ các chữ số đó).
2. Tìm vị trí `i` đầu tiên mà `s[i] != t[i]`. Nếu không có: `num` đã là lớn nhất → trả về `num`.
3. Trước `i`, `s` và `t` giống nhau nên tập chữ số của `s[i:]` bằng tập của `t[i:]`; vì thế chữ số lớn nhất trong `s[i:]` chính là `t[i]`. Đem nó từ vị trí **cuối cùng** (`rindex`) của nó lên `i`.

Cách này cho ta một cách nhìn khác: "vị trí đầu tiên lệch khỏi dãy lý tưởng".

**Code demo:**

In [16]:
def maximum_swap_sorted(num):
    s = list(str(num))
    t = sorted(s, reverse=True)
    for i in range(len(s)):
        if s[i] != t[i]:
            j = len(s) - 1 - s[::-1].index(t[i])    # chỉ số xuất hiện CUỐI của t[i] trong s
            s[i], s[j] = s[j], s[i]
            break
    return int("".join(s))

for num, expected in examples:
    print(maximum_swap_sorted(num), expected)
print(maximum_swap_sorted(1993), maximum_swap_sorted(98368))

7236 7236
9973 9973
9913 98863


**Code nộp LeetCode (Cách 5):**

In [17]:
class SolutionSorted:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        t = sorted(s, reverse=True)
        for i in range(len(s)):
            if s[i] != t[i]:
                j = "".join(s).rindex(t[i])          # lần xuất hiện cuối
                s[i], s[j] = s[j], s[i]
                break
        return int("".join(s))

print(SolutionSorted().maximumSwap(98368))   # 98863

98863


**Độ phức tạp (Cách 5):** Time `O(d log d)` do sắp xếp (với `d ≤ 9` không đáng kể), Space `O(d)`.

## 8. TRACE từng bước

Trace Cách 3 (mảng `d`) cho `num = 2736`, `s = ['2','7','3','6']`:

| i | s[i] | d[i+1] | s[d[i+1]] | s[i] > s[d[i+1]]? | d[i] |
|---|---|---|---|---|---|
| 3 | 6 | — | — | (mặc định) | 3 |
| 2 | 3 | 3 | 6 | không | 3 |
| 1 | 7 | 3 | 6 | có | 1 |
| 0 | 2 | 1 | 7 | không | 1 |

Duyệt trái sang: `i = 0`: `s[0] = 2 < s[d[0]] = s[1] = 7` → đổi vị trí 0 và 1 → `7236`.

Hàm `trace_swap` dưới đây in bảng `d` và quá trình tìm cặp cần đổi cho số bất kỳ.

In [18]:
def trace_swap(num):
    s = list(str(num))
    n = len(s)
    d = list(range(n))
    print(f"num = {num}, s = {s}")
    print("-- Pha 1: dựng d từ phải sang trái --")
    print(f"i={n-1}: d[{n-1}] = {n-1}")
    for i in range(n - 2, -1, -1):
        better = s[i] > s[d[i + 1]]
        d[i] = i if better else d[i + 1]
        print(f"i={i}: s[i]={s[i]} so với s[d[i+1]]={s[d[i+1]]} -> {'lớn hơn, d[i]=i' if better else 'không lớn hơn, d[i]=d[i+1]'} = {d[i]}")
    print("d =", d)
    print("-- Pha 2: tìm vị trí trái nhất còn nâng được --")
    for i in range(n):
        j = d[i]
        print(f"i={i}: s[i]={s[i]}, chữ số lớn nhất bên phải (kể cả i) = s[{j}] = {s[j]}", end=" ")
        if s[i] < s[j]:
            print("-> NÂNG ĐƯỢC, đổi", i, "với", j)
            s[i], s[j] = s[j], s[i]
            break
        print("-> không nâng được")
    else:
        print("không có đổi chỗ nào")
    result = int("".join(s))
    print("kết quả:", result)
    return result

trace_swap(2736)

num = 2736, s = ['2', '7', '3', '6']
-- Pha 1: dựng d từ phải sang trái --
i=3: d[3] = 3
i=2: s[i]=3 so với s[d[i+1]]=6 -> không lớn hơn, d[i]=d[i+1] = 3
i=1: s[i]=7 so với s[d[i+1]]=6 -> lớn hơn, d[i]=i = 1
i=0: s[i]=2 so với s[d[i+1]]=7 -> không lớn hơn, d[i]=d[i+1] = 1
d = [1, 1, 3, 3]
-- Pha 2: tìm vị trí trái nhất còn nâng được --
i=0: s[i]=2, chữ số lớn nhất bên phải (kể cả i) = s[1] = 7 -> NÂNG ĐƯỢC, đổi 0 với 1
kết quả: 7236


7236

Trace thứ hai: `1993` (trùng chữ số lớn nhất) và `98368` (chữ số đầu đã lớn nhất, cần nâng ở vị trí sau). Chú ý cách `d` chọn chỉ số **ngoài cùng bên phải** khi bằng nhau.

In [19]:
trace_swap(1993)
print()
trace_swap(98368)
print()
trace_swap(9973)

num = 1993, s = ['1', '9', '9', '3']
-- Pha 1: dựng d từ phải sang trái --
i=3: d[3] = 3
i=2: s[i]=9 so với s[d[i+1]]=3 -> lớn hơn, d[i]=i = 2
i=1: s[i]=9 so với s[d[i+1]]=9 -> không lớn hơn, d[i]=d[i+1] = 2
i=0: s[i]=1 so với s[d[i+1]]=9 -> không lớn hơn, d[i]=d[i+1] = 2
d = [2, 2, 2, 3]
-- Pha 2: tìm vị trí trái nhất còn nâng được --
i=0: s[i]=1, chữ số lớn nhất bên phải (kể cả i) = s[2] = 9 -> NÂNG ĐƯỢC, đổi 0 với 2
kết quả: 9913

num = 98368, s = ['9', '8', '3', '6', '8']
-- Pha 1: dựng d từ phải sang trái --
i=4: d[4] = 4
i=3: s[i]=6 so với s[d[i+1]]=8 -> không lớn hơn, d[i]=d[i+1] = 4
i=2: s[i]=3 so với s[d[i+1]]=8 -> không lớn hơn, d[i]=d[i+1] = 4
i=1: s[i]=8 so với s[d[i+1]]=8 -> không lớn hơn, d[i]=d[i+1] = 4
i=0: s[i]=9 so với s[d[i+1]]=8 -> lớn hơn, d[i]=i = 0
d = [0, 4, 4, 4, 4]
-- Pha 2: tìm vị trí trái nhất còn nâng được --
i=0: s[i]=9, chữ số lớn nhất bên phải (kể cả i) = s[0] = 9 -> không nâng được
i=1: s[i]=8, chữ số lớn nhất bên phải (kể cả i) = s[4] = 8 -> không nâng

9973

Trace Cách 4 (một lượt phải sang trái) cho `98368`: in `maxi`, ứng viên `(l, r)` qua từng bước.

In [20]:
def trace_one_pass(num):
    s = list(str(num))
    n = len(s)
    maxi = n - 1
    l = r = -1
    print(f"s = {s}; bắt đầu maxi = {maxi} (chữ số {s[maxi]})")
    for i in range(n - 2, -1, -1):
        if s[i] > s[maxi]:
            action = f"lớn hơn s[maxi]={s[maxi]} -> maxi = {i}"
            maxi = i
        elif s[i] < s[maxi]:
            action = f"nhỏ hơn s[maxi]={s[maxi]} -> ứng viên (l, r) = ({i}, {maxi})"
            l, r = i, maxi
        else:
            action = "bằng s[maxi] -> giữ nguyên"
        print(f"i={i}: s[i]={s[i]} {action}")
    if l != -1:
        s[l], s[r] = s[r], s[l]
        print(f"đổi vị trí {l} và {r}")
    else:
        print("không đổi")
    print("kết quả:", int("".join(s)))

trace_one_pass(98368)

s = ['9', '8', '3', '6', '8']; bắt đầu maxi = 4 (chữ số 8)
i=3: s[i]=6 nhỏ hơn s[maxi]=8 -> ứng viên (l, r) = (3, 4)
i=2: s[i]=3 nhỏ hơn s[maxi]=8 -> ứng viên (l, r) = (2, 4)
i=1: s[i]=8 bằng s[maxi] -> giữ nguyên
i=0: s[i]=9 lớn hơn s[maxi]=8 -> maxi = 0
đổi vị trí 2 và 4
kết quả: 98863


## 9. Code nộp LeetCode

Lời giải chuẩn: một lượt duyệt phải-sang-trái (Cách 4) — ngắn, `O(d)`, không mảng phụ. Đúng `class Solution` và signature.

In [21]:
from typing import List

class Solution:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        n = len(s)
        maxi = n - 1               # chỉ số chữ số lớn nhất bên phải (ưu tiên ngoài cùng bên phải)
        l = r = -1                 # cặp cần đổi
        for i in range(n - 2, -1, -1):
            if s[i] > s[maxi]:
                maxi = i
            elif s[i] < s[maxi]:
                l, r = i, maxi     # nâng được; i càng về trái càng tốt
        if l != -1:
            s[l], s[r] = s[r], s[l]
        return int("".join(s))

sol = Solution()
for num, expected in examples:
    got = sol.maximumSwap(num)
    print(got, expected, "OK" if got == expected else "SAI")

7236 7236 OK
9973 9973 OK


## 10. Vì sao đúng

Gọi `s` là chuỗi chữ số, độ dài `d`. Ta chứng minh quy tắc ở mục 6 cho đáp án tối ưu.

**Trường hợp 1: không có `i` nào để nâng.** Tức mọi `s[i] ≥ s[j]` với `j > i`: dãy không tăng. Với mọi cặp `i < j` đổi chỗ cho ra `s[i]` mới `= s[j] ≤ s[i]`; nếu bằng nhau thì số không đổi, nếu nhỏ hơn thì số giảm (chữ số ở vị trí cao hơn giảm). Vậy không đổi là tối ưu.

**Trường hợp 2: có `i` nâng được.** Gọi `i*` là vị trí **trái nhất** như vậy (nâng được).
- Mọi đổi chỗ chỉ liên quan vị trí `p < i*` hoặc `q > p` bất kỳ: với `p < i*`, `s[p]` ≥ mọi chữ số bên phải, nên đổi chỗ `p` không tăng được chữ số ở `p` (bằng hoặc giảm) — nếu giảm thì số giảm; nếu bằng thì số không đổi. Do đó đổi chỗ **tốt nhất** phải có vị trí trái hơn là `≥ i*`; đổi ở `p > i*` thì chữ số ở `i*` không tăng, nên kém hơn đổi tại `i*` (chữ số ở `i*` tăng thành `M > s[i*]`).
- Tại `i*`, chữ số đem lên càng lớn càng tốt → `M = max(s[i*+1:])`.
- Nếu `M` xuất hiện nhiều lần ở các chỉ số `j1 < j2 < ...`, kết quả khi đổi `i*` với `jk`: các chữ số ở vị trí `j ≠ jk` cùng nhóm vẫn là `M`, chữ số ở `jk` thành `s[i*]` (nhỏ hơn `M`). Mọi vị trí khác `i*, jk` giữ nguyên. So hai lựa chọn `ja < jb`: ở vị trí `ja`, lựa chọn `a` có `s[i*]`, lựa chọn `b` có `M > s[i*]` → `b` lớn hơn (vì mọi vị trí trước `ja` giống nhau). Do đó chọn `jk` **lớn nhất**.

**Code làm đúng như vậy:** duyệt từ phải sang, `maxi` luôn là chỉ số ngoài cùng bên phải của chữ số lớn nhất bên phải `i` (dấu `>` chặt nên khi bằng nhau không đổi `maxi`); ghi `l, r` mỗi khi `s[i] < s[maxi]`, nên giá trị cuối là `i` **trái nhất** còn nâng được.

Cell dưới kiểm chứng bằng **vét cạn mọi số tới 200.000** (đối chiếu Solution với brute force).

In [22]:
bad = 0
for num in range(0, 200001):
    if Solution().maximumSwap(num) != maximum_swap_brute(num):
        bad += 1
        print("lệch ở", num)
        break
print("vét cạn 0..200000: số trường hợp lệch =", bad)

vét cạn 0..200000: số trường hợp lệch = 0


## 11. Time / Space Complexity

Gọi `d = số chữ số của num = O(log num)` (tối đa 9 theo đề).

- **Time:** `O(d)` — chuyển chuỗi `O(d)`, một vòng lặp `O(d)`, ghép lại `O(d)`.
- **Space:** `O(d)` — danh sách chữ số (chuỗi trong Python không sửa được). Nếu cho phép dùng mảng chữ số làm đầu vào thì phần thuật toán là `O(1)`.

Nếu `num` rất lớn (hàng nghìn chữ số, biểu diễn bằng chuỗi), cách này vẫn `O(d)` còn brute force `O(d^3)`.

In [23]:
# Số 4000 chữ số: tham lam O(d) chạy ngay; brute O(d^3) sẽ rất lâu nên ta không chạy
import time
random.seed(670)
big = random.choice("123456789") + "".join(random.choice("0123456789") for _ in range(3999))
t0 = time.perf_counter()
r1 = SolutionOnePass().maximumSwap(int(big))
r2 = SolutionLast().maximumSwap(int(big))
r3 = SolutionRightMax().maximumSwap(int(big))
r4 = SolutionSorted().maximumSwap(int(big))
assert r1 == r2 == r3 == r4
print("4 cách tham lam khớp nhau trên số", len(big), "chữ số; thời gian < 5 giây:", time.perf_counter() - t0 < 5)

4 cách tham lam khớp nhau trên số 4000 chữ số; thời gian < 5 giây: True


## 12. Bộ test

Gồm: ví dụ đề, edge case, phản ví dụ cho các cách sai, đối chiếu các cách bằng `assert`, và test ngẫu nhiên có seed trong ràng buộc.

In [24]:
sol = Solution()

# 1) Ví dụ đề
assert sol.maximumSwap(2736) == 7236
assert sol.maximumSwap(9973) == 9973
print("ví dụ đề OK")

# 2) Edge case
assert sol.maximumSwap(0) == 0                  # nhỏ nhất
assert sol.maximumSwap(7) == 7                  # một chữ số
assert sol.maximumSwap(10) == 10                # đổi sẽ ra 01 = 1, nhỏ hơn
assert sol.maximumSwap(12) == 21
assert sol.maximumSwap(98) == 98
assert sol.maximumSwap(111) == 111              # toàn chữ số giống nhau
assert sol.maximumSwap(100000000) == 100000000  # 10^8: đổi chỗ 1 với 0 chỉ làm nhỏ đi
assert sol.maximumSwap(99999999) == 99999999
assert sol.maximumSwap(1234) == 4231            # tăng dần: đổi đầu - cuối
assert sol.maximumSwap(109) == 901              # 0 không được đưa lên đầu, nhưng 9 lên đầu
assert sol.maximumSwap(1993) == 9913            # trùng: lấy 9 cuối cùng
assert sol.maximumSwap(98368) == 98863          # đầu đã lớn nhất; nâng ở vị trí sau
assert sol.maximumSwap(1099) == 9091            # 1 đổi với 9 cuối cùng
print("edge case OK")

ví dụ đề OK
edge case OK


In [25]:
# 3) Phản ví dụ cho các cách sai (xem mục 13)
assert maximum_swap_brute(1993) == 9913 and maximum_swap_brute(98368) == 98863 and maximum_swap_brute(9368) == 9863
assert maximum_swap_brute(1099) == sol.maximumSwap(1099)
print("phản ví dụ OK; 1099 ->", sol.maximumSwap(1099))

# 4) Đối chiếu 5 cách + brute force trên mọi số tới 20000 và một số mốc đặc biệt
solutions = [SolutionLast(), SolutionRightMax(), SolutionOnePass(), SolutionSorted(), Solution()]
funcs = [maximum_swap_last, maximum_swap_rightmax, maximum_swap_one_pass, maximum_swap_sorted]
for num in list(range(0, 20001)) + [99999999, 100000000, 12345678, 87654321, 11223344, 90909090]:
    expected = maximum_swap_brute(num)
    for f in funcs:
        assert f(num) == expected, (f.__name__, num)
    for s_ in solutions:
        assert s_.maximumSwap(num) == expected, (type(s_).__name__, num)
print("20001 số + mốc đặc biệt: mọi cách khớp brute force")

phản ví dụ OK; 1099 -> 9091


20001 số + mốc đặc biệt: mọi cách khớp brute force


In [26]:
# 5) Ngẫu nhiên có seed, đúng ràng buộc 0 <= num <= 10^8
random.seed(2026)
for _ in range(3000):
    num = random.randint(0, 10**8)
    expected = maximum_swap_brute(num)
    for s_ in solutions:
        assert s_.maximumSwap(num) == expected, (type(s_).__name__, num)
# nhấn mạnh trường hợp có nhiều chữ số lặp (ít chữ số khác nhau)
for _ in range(3000):
    k = random.randint(1, 3)
    pool = random.sample("0123456789", k)
    length = random.randint(1, 9)
    digits = [random.choice(pool) for _ in range(length)]
    if digits[0] == "0" and length > 1:
        digits[0] = random.choice([p for p in pool if p != "0"] or ["1"])
    num = int("".join(digits))
    if num > 10**8:
        continue
    expected = maximum_swap_brute(num)
    for s_ in solutions:
        assert s_.maximumSwap(num) == expected, (type(s_).__name__, num)
print("6000 số ngẫu nhiên: khớp")

# 6) Test tính chất: kết quả >= num, cùng tập chữ số, và khác num tối đa ở 2 vị trí
for _ in range(2000):
    num = random.randint(0, 10**8)
    res = sol.maximumSwap(num)
    a, b = str(num), str(res)
    assert res >= num
    assert sorted(a) == sorted(b) and len(a) == len(b)
    assert sum(x != y for x, y in zip(a, b)) in (0, 2)
print("test tính chất OK")

6000 số ngẫu nhiên: khớp
test tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chọn lần xuất hiện **đầu tiên** của chữ số lớn nhất

Dùng `index` thay vì `rindex`, hoặc dùng `>=` thay vì `>` trong mảng `d`.

In [27]:
def wrong_first_occurrence(num):
    s = list(str(num))
    for i in range(len(s)):
        right = s[i + 1:]
        if right and max(right) > s[i]:
            j = i + 1 + right.index(max(right))        # SAI: lần xuất hiện đầu tiên
            s[i], s[j] = s[j], s[i]
            break
    return int("".join(s))

print("sai :", wrong_first_occurrence(1993), "(kỳ vọng 9913)")
print("đúng:", Solution().maximumSwap(1993))

sai : 9193 (kỳ vọng 9913)
đúng: 9913


**Vì sao sai:** đổi `1` với `9` đầu tiên cho `9193`; đổi với `9` cuối cho `9913`. Chữ số nhỏ bị đẩy xuống nên phải đẩy xuống vị trí **thấp nhất**, tức ngoài cùng bên phải. **Cách nhớ:** *chữ số bé nhất ném càng xa càng tốt*.

### Lỗi 2 — Sắp xếp giảm dần toàn bộ

Nhầm "số lớn nhất" với "số lớn nhất ghép từ các chữ số" — nhưng đề chỉ cho đổi chỗ **một lần**.

In [28]:
def wrong_sort_all(num):
    return int("".join(sorted(str(num), reverse=True)))      # SAI: cần nhiều lần đổi chỗ

print("sai :", wrong_sort_all(2736), "(kỳ vọng 7236)")
print("đúng:", Solution().maximumSwap(2736))

sai : 7632 (kỳ vọng 7236)
đúng: 7236


**Vì sao sai:** `7632` cần đổi chỗ hai lần (`2↔7`, rồi `2↔6`... để đưa `6` lên). Đề giới hạn tối đa **một** lần.

### Lỗi 3 — Chỉ so với chữ số lớn nhất toàn cục (bỏ qua nâng ở vị trí sau)

Nghĩ rằng "lấy chữ số lớn nhất của cả số đem lên đầu". Nếu chữ số đầu **đã** là lớn nhất, ta không đổi gì — nhưng đáng ra phải nâng ở vị trí kế tiếp.

In [29]:
def wrong_global_max(num):
    s = list(str(num))
    m = max(s)
    j = len(s) - 1 - s[::-1].index(m)       # lần cuối của chữ số lớn nhất toàn cục
    if s[0] != m:
        s[0], s[j] = s[j], s[0]             # SAI: chỉ xét vị trí 0
    return int("".join(s))

print("sai :", wrong_global_max(9368), "(kỳ vọng 9863)")
print("đúng:", Solution().maximumSwap(9368))
print("sai :", wrong_global_max(98368), "(kỳ vọng 98863)")
print("đúng:", Solution().maximumSwap(98368))

sai : 9368 (kỳ vọng 9863)
đúng: 9863
sai : 98368 (kỳ vọng 98863)
đúng: 98863


**Vì sao sai:** vị trí đầu `9` đã tối ưu, nhưng vị trí kế `3` vẫn nâng được lên `8`. **Quy tắc:** tìm vị trí **trái nhất còn nâng được**, rồi mới chọn chữ số lớn nhất **bên phải vị trí đó**.

### Lỗi 4 — Không dừng sau lần đổi đầu tiên

Đổi chỗ tiếp ở các vị trí sau, vi phạm "tối đa một lần".

In [30]:
def wrong_no_break(num):
    s = list(str(num))
    for i in range(len(s)):
        right = s[i + 1:]
        if right and max(right) > s[i]:
            j = i + 1 + len(right) - 1 - right[::-1].index(max(right))
            s[i], s[j] = s[j], s[i]              # SAI: thiếu break -> đổi nhiều lần
    return int("".join(s))

print("sai :", wrong_no_break(2736), "(kỳ vọng 7236)")
print("đúng:", Solution().maximumSwap(2736))

sai : 7632 (kỳ vọng 7236)
đúng: 7236


**Vì sao sai:** sau khi đổi `2↔7` ta vẫn tiếp tục và đổi `2↔6`,... kết quả `7632` — hai lần đổi chỗ.

### Lỗi 5 — Trả về chuỗi hoặc quên `int(...)`

Hàm cần trả về `int`. Nếu trả `"".join(s)` bạn nhận chuỗi `"7236"`, và `"7236" == 7236` là `False` — test `assert` sẽ báo sai dù nhìn giống đúng.

In [31]:
s = list("2736")
s[0], s[1] = s[1], s[0]
wrong_result = "".join(s)
print(repr(wrong_result), wrong_result == 7236)        # '7236' False
print(repr(int(wrong_result)), int(wrong_result) == 7236)

'7236' False
7236 True


**Cách nhớ:** hàm trả về số → kết thúc bằng `int(...)`.

## 14. Mẹo & nhận dạng pattern

- **"Đổi chỗ / chọn tối đa k lần để số lớn nhất"** → tham lam theo trọng số vị trí: chữ số trái có giá trị lớn hơn mọi chữ số phía sau cộng lại (về mặt so sánh). Xử lý **từ trái sang**, sửa tại chỗ đầu tiên có lợi.
- **Trùng giá trị** là chỗ dễ sai: hỏi "bé đi xuống đâu thì ít hại nhất?" → vị trí thấp nhất (ngoài cùng bên phải).
- **Duyệt phải sang trái** với biến "lớn nhất đã thấy bên phải" thay cho mảng hậu tố: kỹ thuật gọn thường gặp (cùng họ với tìm phần tử lớn hơn bên phải, Next Greater, trapping rain...).
- **Bảng vị trí cuối 0..9:** khi tập giá trị nhỏ (chữ số), bảng 10 ô thay cho cấu trúc tìm kiếm.
- **Kiểm chứng:** viết brute force "thử mọi cặp" và vét cạn các số nhỏ — rẻ và bắt được mọi lỗi tham lam.
- Bài họ hàng: Maximum Swap k lần (BFS/DFS), Remove K Digits (402), Next Permutation (31), Create Maximum Number (321).

## 15. Biến thể

### 15.1. Trả về cặp vị trí cần đổi (hoặc `None`)

Chỉ cần trả `(l, r)` thay vì số mới.

In [32]:
def best_swap_positions(num):
    s = str(num)
    n = len(s)
    maxi = n - 1
    l = r = -1
    for i in range(n - 2, -1, -1):
        if s[i] > s[maxi]:
            maxi = i
        elif s[i] < s[maxi]:
            l, r = i, maxi
    return (l, r) if l != -1 else None

print(best_swap_positions(2736))    # (0, 1)
print(best_swap_positions(9973))    # None
print(best_swap_positions(1993))    # (0, 2)

(0, 1)
None
(0, 2)


### 15.2. Số **nhỏ nhất** sau tối đa một lần đổi chỗ (không cho số 0 đứng đầu)

Đối xứng với bài gốc, nhưng có bẫy: **không được đưa chữ số `0` lên đầu** (số có số 0 đứng đầu không hợp lệ). Tham lam: duyệt `i` từ trái sang; tìm chữ số **nhỏ nhất** bên phải `i` (ở `i = 0` thì chỉ xét chữ số **khác 0**); nếu nó nhỏ hơn `s[i]` thì đổi với **lần xuất hiện cuối** của nó và dừng, ngược lại sang `i` kế tiếp.

In [33]:
def minimum_swap_brute(num):
    s = str(num)
    best = num
    for i, j in combinations(range(len(s)), 2):
        t = list(s)
        t[i], t[j] = t[j], t[i]
        if t[0] == "0" and len(t) > 1:
            continue                              # không cho số 0 đứng đầu
        best = min(best, int("".join(t)))
    return best

def minimum_swap(num):
    s = list(str(num))
    n = len(s)
    for i in range(n - 1):
        cand = [c for c in s[i + 1:] if not (i == 0 and c == "0")]
        if not cand:
            continue
        m = min(cand)
        if m < s[i]:
            j = n - 1 - s[::-1].index(m)          # lần xuất hiện cuối của m
            s[i], s[j] = s[j], s[i]
            break
    return int("".join(s))

print(minimum_swap(2736), minimum_swap(9973), minimum_swap(1993), minimum_swap(100), minimum_swap(5093))
for num in range(0, 100001):
    assert minimum_swap(num) == minimum_swap_brute(num), num
print("số nhỏ nhất: khớp brute force trên 0..100000")

2376 3979 1399 100 3095


số nhỏ nhất: khớp brute force trên 0..100000


### 15.3. Tối đa `k` lần đổi chỗ

Với `k ≥ 2` tham lam đơn giản không còn chắc đúng (cần thử các lựa chọn khi chữ số lớn nhất trùng nhau). Cách an toàn: **BFS theo tầng** — mỗi tầng là tập các số đạt được sau thêm một lần đổi; lấy lớn nhất trong tất cả các tầng ≤ `k`. Số trạng thái tối đa là số hoán vị phân biệt của các chữ số, nên chỉ dùng khi số chữ số nhỏ.

In [34]:
def max_with_k_swaps(num, k):
    s = str(num)
    level = {s}
    best = num
    for _ in range(k):
        nxt = set()
        for cur in level:
            for i, j in combinations(range(len(cur)), 2):
                t = list(cur)
                t[i], t[j] = t[j], t[i]
                nxt.add("".join(t))
        if not nxt:                       # số 1 chữ số: không có cặp để đổi
            break
        level = nxt
        best = max(best, max(int(x) for x in level))
    return best

print(max_with_k_swaps(2736, 1))   # 7236 (bằng đề gốc)
print(max_with_k_swaps(2736, 2))   # 7632
print(max_with_k_swaps(1993, 2))   # 9931
for num in range(0, 3000):
    assert max_with_k_swaps(num, 1) == Solution().maximumSwap(num)
print("k = 1 khớp lời giải chính")

7236
7632
9931
k = 1 khớp lời giải chính


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | Vét cạn `C(d,2)` lần đổi | `O(d^3)` | `O(d)` | Làm oracle kiểm tra |
| 2. Bảng vị trí cuối 0..9 | Với mỗi `i`, tìm chữ số lớn hơn từ 9 xuống | `O(10·d)` | `O(d)` | Dễ hiểu; tận dụng tập chữ số nhỏ |
| 3. Mảng chỉ số lớn nhất bên phải | `d[i]` dựng từ phải sang trái | `O(d)` | `O(d)` | Dạng chuẩn của doocs; mở rộng được cho bài khác |
| 4. Một lượt phải → trái | Giữ `maxi` và cặp `(l, r)` | `O(d)` | `O(1)` phụ | **Lời giải chuẩn nộp LeetCode** |
| 5. So với dãy sắp xếp giảm | Vị trí đầu tiên lệch dãy lý tưởng | `O(d log d)` | `O(d)` | Cách nhìn khác; hữu ích để kiểm tra |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().maximumSwap(2736))
print(Solution().maximumSwap(98368))
print(Solution().maximumSwap(1993))
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `num = 2736` (`s = "2736"`), hãy tính tay mảng `d` ở Cách 3 (`d[i]` = chỉ số chữ số lớn nhất trong `s[i:]`, ưu tiên bên phải khi bằng nhau) và cặp `(i, d[i])` được đổi. Điền vào các biến.

In [36]:
# Bài 2: điền kết quả tính tay
hand_d = None          # ví dụ: [.., .., .., ..]
hand_swap_pair = None  # (i, j) cặp vị trí đổi chỗ

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Cách 2 (bảng vị trí cuối).

In [37]:
def exercise_3_maximum_swap(num: int) -> int:
    s = list(str(num))
    # TODO 1: last = list 10 phần tử -1; duyệt s, ghi last[int(ch)] = i
    # TODO 2: với mỗi i, ch: duyệt d từ 9 xuống int(ch)+1
    #     TODO 3: nếu last[d] > i thì đổi s[i] và s[last[d]] rồi return int(...)
    # TODO 4: nếu không đổi được, return num
    return None


# assert exercise_3_maximum_swap(2736) == 7236
# assert exercise_3_maximum_swap(9973) == 9973
# assert exercise_3_maximum_swap(1993) == 9913

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_one_pass(num)`: một vòng `for` từ phải sang trái, giữ `maxi` và `(l, r)`.

In [38]:
def exercise_4_one_pass(num: int) -> int:
    # TODO
    pass


# assert exercise_4_one_pass(2736) == 7236
# assert exercise_4_one_pass(98368) == 98863
# assert exercise_4_one_pass(10) == 10

### Bài 5 — Mini challenge: số nhỏ nhất sau một lần đổi (khó)

Viết `exercise_5_min_swap(num)` trả về số **nhỏ nhất** thu được khi đổi chỗ hai chữ số tối đa một lần, với điều kiện **kết quả không được có số 0 đứng đầu** (trừ khi bản thân `num` là một chữ số).

```python
exercise_5_min_swap(2736) -> 2376
exercise_5_min_swap(9973) -> 3979
exercise_5_min_swap(100)  -> 100
exercise_5_min_swap(5093) -> 3095
```

In [39]:
def exercise_5_min_swap(num: int) -> int:
    # TODO
    pass


# kết quả được kiểm tra ở mục 19 bằng brute force

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Chữ số bên trái quan trọng hơn. Tìm vị trí trái nhất mà bên phải có chữ số lớn hơn nó.
- *Mức 2 (vừa):* Tại vị trí đó, đổi với chữ số **lớn nhất** bên phải; nếu trùng thì lấy cái **ngoài cùng bên phải**. Sau khi đổi phải dừng.
- *Mức 3 (gần đáp án):* Cách 2: `last[int(ch)] = i` rồi thử `d` từ `9` xuống `int(ch)+1`, `last[d] > i` thì đổi. Cách 4: `maxi = n-1`; duyệt `i` từ `n-2` xuống `0`: `s[i] > s[maxi]` → `maxi = i`; `s[i] < s[maxi]` → `l, r = i, maxi`.

**Bài 5**

- *Mức 1:* Đối xứng với bài gốc: tìm chữ số **nhỏ nhất** bên phải để đem lên vị trí trái nhất mà còn giảm được.
- *Mức 2:* Bẫy: ở vị trí 0 không được đưa chữ số `0` lên. Với `i = 0`, bỏ qua các chữ số `0` khi tìm nhỏ nhất.
- *Mức 3:* Với mỗi `i` từ trái: `cand` = chữ số bên phải (trừ `'0'` khi `i == 0`); nếu `min(cand) < s[i]` thì đổi `i` với **lần cuối** của `min(cand)` và dừng.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- `2736` → `7236`.
- `98368` → `98863` (đổi `3` và `8` cuối).
- `1993` → `9913`.

In [40]:
assert Solution().maximumSwap(2736) == 7236
assert Solution().maximumSwap(98368) == 98863
assert Solution().maximumSwap(1993) == 9913
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

`s = 2,7,3,6`. `d[3] = 3`; `d[2]`: `3 > 6`? không → `3`; `d[1]`: `7 > s[3] = 6` → `1`; `d[0]`: `2 > s[1] = 7`? không → `1`. Vậy `d = [1, 1, 3, 3]`. Vị trí 0 là trái nhất có `s[0] = 2 < s[d[0]] = 7` → đổi `(0, 1)`.

In [41]:
hand_d = [1, 1, 3, 3]
hand_swap_pair = (0, 1)

s_ = list("2736")
n_ = len(s_)
d_ = list(range(n_))
for i in range(n_ - 2, -1, -1):
    d_[i] = i if s_[i] > s_[d_[i + 1]] else d_[i + 1]
assert hand_d == d_
assert hand_swap_pair == best_swap_positions(2736)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(num: int) -> int:
    s = list(str(num))
    last = [-1] * 10
    for i, ch in enumerate(s):
        last[int(ch)] = i
    for i, ch in enumerate(s):
        for d in range(9, int(ch), -1):
            if last[d] > i:
                s[i], s[last[d]] = s[last[d]], s[i]
                return int("".join(s))
    return num

assert exercise_3_solution(2736) == 7236
assert exercise_3_solution(9973) == 9973
assert exercise_3_solution(1993) == 9913
for num in range(0, 5000):
    assert exercise_3_solution(num) == maximum_swap_brute(num)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(num: int) -> int:
    s = list(str(num))
    n = len(s)
    maxi = n - 1
    l = r = -1
    for i in range(n - 2, -1, -1):
        if s[i] > s[maxi]:
            maxi = i
        elif s[i] < s[maxi]:
            l, r = i, maxi
    if l != -1:
        s[l], s[r] = s[r], s[l]
    return int("".join(s))

assert exercise_4_solution(2736) == 7236
assert exercise_4_solution(98368) == 98863
assert exercise_4_solution(10) == 10
for num in range(0, 5000):
    assert exercise_4_solution(num) == maximum_swap_brute(num)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Số nhỏ nhất: với `2736` vị trí 0 là `2`, các chữ số bên phải `7,3,6` đều lớn hơn `2` nên không giảm được ở vị trí 0; sang vị trí 1 (`7`): bên phải có `3` và `6`, nhỏ nhất `3 < 7` → đổi `7` với `3` → `2376`. Các ví dụ còn lại: `9973 → 3979` (đổi `9` đầu với `3`), `100 → 100` (không được đưa `0` lên đầu), `5093 → 3095`.

In [44]:
def exercise_5_solution(num: int) -> int:
    s = list(str(num))
    n = len(s)
    for i in range(n - 1):
        cand = [c for c in s[i + 1:] if not (i == 0 and c == "0")]
        if not cand:
            continue
        m = min(cand)
        if m < s[i]:
            j = n - 1 - s[::-1].index(m)
            s[i], s[j] = s[j], s[i]
            break
    return int("".join(s))

assert exercise_5_solution(2736) == 2376
assert exercise_5_solution(9973) == 3979
assert exercise_5_solution(100) == 100
assert exercise_5_solution(5093) == 3095
for num in range(0, 100001):
    assert exercise_5_solution(num) == minimum_swap_brute(num), num
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def maximumSwap(self, num: int) -> int:
        s = list(str(num))
        n = len(s)
        maxi = n - 1                       # chữ số lớn nhất bên phải (ưu tiên ngoài cùng bên phải)
        l = r = -1
        for i in range(n - 2, -1, -1):
            if s[i] > s[maxi]:
                maxi = i
            elif s[i] < s[maxi]:
                l, r = i, maxi             # nâng được; i càng về trái càng tốt
        if l != -1:
            s[l], s[r] = s[r], s[l]
        return int("".join(s))
```

Câu thần chú: **"Từ phải sang trái nhớ chữ số lớn nhất; gặp chữ số bé hơn thì ghi cặp; cặp cuối cùng (trái nhất) là cặp đổi."**

### Checklist trước khi submit

- [ ] Chọn vị trí **trái nhất** còn nâng được; chữ số đem lên là **lớn nhất** bên phải.
- [ ] Khi trùng chữ số lớn nhất: lấy lần xuất hiện **cuối cùng** (dùng `>` chặt khi cập nhật `maxi`).
- [ ] Đổi **một lần** rồi dừng (không đổi tiếp).
- [ ] Chuyển sang `list` mới sửa được; trả về `int(...)`.
- [ ] Đã thử `0`, một chữ số, `10`, dãy giảm dần (không đổi), dãy tăng dần, chữ số lặp (`1993`), `10^8`.
- [ ] Đã đối chiếu brute force trên mọi số nhỏ.
- [ ] Skeleton đúng: `class Solution`, method `maximumSwap(self, num: int) -> int`.